# Tweak generated card features

Controls use narrow ranges estimated from the reference photos. Height follows width/height aspect ratio; symbol spacing uses a fixed 24% of card height with small variation. Both the rectangle/ellipse deck and the diamond/capsule deck are supported.


In [ ]:
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from set_master.generation.card import draw_card
from set_master.generation.features import FeatureStyle
from set_master.models import Card, Color, Number, Shading, ShadingPattern, Shape
from set_master.visualization import display_image

shape = widgets.Dropdown(options=[(item.name.title(), item) for item in Shape], description='Shape')
shading = widgets.Dropdown(options=[(item.name.title(), item) for item in Shading], description='Shading')
color = widgets.Dropdown(options=[(item.name.title(), item) for item in Color], description='Color')
number = widgets.Dropdown(options=[(item.name.title(), item) for item in Number], description='Count')
card_width = widgets.IntSlider(value=360, min=180, max=600, step=10, description='Width (px)', continuous_update=False)
card_aspect_ratio = widgets.FloatSlider(value=2/3, min=0.64, max=0.69, step=0.005, description='Width/height', readout_format='.3f', continuous_update=False)
corner_radius_ratio = widgets.FloatSlider(value=0.045, min=0.035, max=0.055, step=0.001, description='Corner/width', readout_format='.3f', continuous_update=False)
feature_width_ratio = widgets.FloatSlider(value=0.64, min=0.60, max=0.68, step=0.01, description='Symbol/width', continuous_update=False)
feature_height_ratio = widgets.FloatSlider(value=0.18, min=0.16, max=0.20, step=0.01, description='Symbol/height', continuous_update=False)
symbol_spacing_ratio = widgets.FloatSlider(value=1.0, min=0.95, max=1.05, step=0.01, description='Spacing scale', continuous_update=False)
shading_pattern = widgets.Dropdown(options=[(item.value.replace('_', ' ').title(), item) for item in ShadingPattern], description='Pattern')
stripe_spacing = widgets.IntSlider(value=5, min=4, max=6, description='Pattern spacing', continuous_update=False)
line_thickness = widgets.IntSlider(value=2, min=1, max=2, description='Line thickness', continuous_update=False)
squiggle_amplitude = widgets.FloatSlider(value=0.18, min=0.16, max=0.20, step=0.01, description='Curve offset', continuous_update=False)
squiggle_waves = widgets.FloatSlider(value=1.0, min=0.95, max=1.05, step=0.01, description='Curve cycles', continuous_update=False)
squiggle_thickness = widgets.FloatSlider(value=0.65, min=0.60, max=0.70, step=0.01, description='Body thickness', continuous_update=False)

layout_controls = widgets.VBox([shape, shading, color, number, card_width, card_aspect_ratio, corner_radius_ratio, feature_width_ratio, feature_height_ratio, symbol_spacing_ratio])
pattern_controls = widgets.VBox([shading_pattern, stripe_spacing, line_thickness])
squiggle_controls = widgets.VBox([squiggle_amplitude, squiggle_waves, squiggle_thickness])
groups = widgets.Accordion(children=[layout_controls, pattern_controls, squiggle_controls])
for index, title in enumerate(['Card and symbols', 'Shaded patterns and outlines', 'Squiggle geometry']):
    groups.set_title(index, title)

def render(shape, shading, color, number, card_width, card_aspect_ratio, corner_radius_ratio, feature_width_ratio, feature_height_ratio, symbol_spacing_ratio, shading_pattern, stripe_spacing, line_thickness, squiggle_amplitude, squiggle_waves, squiggle_thickness):
    card_height = round(card_width / card_aspect_ratio)
    canvas = np.full((card_height + 24, card_width + 24, 3), 245, dtype=np.uint8)
    style = FeatureStyle(
        shading_pattern=shading_pattern, stripe_spacing=stripe_spacing,
        line_thickness=line_thickness, squiggle_amplitude=squiggle_amplitude,
        squiggle_waves=squiggle_waves, squiggle_thickness=squiggle_thickness,
    )
    draw_card(
        canvas, Card(shape=shape, color=color, shading=shading, number=number),
        size=(card_width, card_height),
        feature_size=(round(card_width * feature_width_ratio), round(card_height * feature_height_ratio)),
        symbol_spacing=round(card_height * 0.24 * symbol_spacing_ratio),
        corner_radius=round(card_width * corner_radius_ratio), feature_style=style,
    )
    display_image(canvas)

controls = {name: value for name, value in locals().copy().items() if name in render.__code__.co_varnames[:render.__code__.co_argcount]}
preview = widgets.interactive_output(render, controls)
display(widgets.VBox([groups, preview]))
